# PHÂN TÍCH HÀNH VI TOXIC TRONG BÌNH LUẬN MẠNG XÃ HỘI TIẾNG VIỆT

---

## Mô tả dự án

Dự án xây dựng pipeline thu thập, làm sạch, gán nhãn và phân tích bình luận độc hại (toxic) trên mạng xã hội tiếng Việt. Hệ thống kết hợp:

- **Dataset ViHSD** (Vietnamese Hate Speech Detection) — tập dữ liệu benchmark có nhãn sẵn
- **Dữ liệu thu thập** từ YouTube, Reddit, Facebook thông qua các scraper tự viết
- **Gemini AI** để tự động gán nhãn dữ liệu mạng xã hội chưa có nhãn

**Phân loại nhãn:**
| Nhãn | Ý nghĩa |
|------|----------|
| `CLEAN` | Bình luận bình thường, không có nội dung tiêu cực |
| `OFFENSIVE` | Bình luận xúc phạm, thô lỗ |
| `HATE` | Bình luận thù ghét, kích động bạo lực |

---

## Pipeline 5 bước

```
BƯỚC 1  →  BƯỚC 2  →  BƯỚC 3  →  BƯỚC 4  →  BƯỚC 5
Thu thập    Làm sạch    Gán nhãn    Thống kê    Trực quan
dữ liệu     dữ liệu     Gemini AI   & Phân tích  hóa
```

## 0. Cài đặt & Cấu hình

In [1]:
# Cài đặt thư viện cần thiết
# !pip install -r requirements.txt

In [2]:
import os
import sys
import json
import warnings
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import seaborn as sns
from pathlib import Path
from dotenv import load_dotenv
from IPython.display import Image, display

warnings.filterwarnings('ignore')

# ── Cấu hình đường dẫn ──────────────────────────────────────────────────────
BASE_DIR   = Path('.')          # Thư mục toxic_analysis/
ROOT_DIR   = BASE_DIR.parent
VIHSD_DIR  = ROOT_DIR / 'vihsd'
DATA_DIR   = BASE_DIR / 'data' / 'collected'
OUTPUT_DIR = BASE_DIR / 'output'
CHARTS_DIR = OUTPUT_DIR / 'charts'

sys.path.insert(0, str(BASE_DIR))
load_dotenv(ROOT_DIR / '.env')

# ── Cấu hình hiển thị ───────────────────────────────────────────────────────
pd.set_option('display.max_columns', 10)
pd.set_option('display.max_colwidth', 60)
pd.set_option('display.float_format', '{:.2f}'.format)

matplotlib.rcParams['figure.dpi'] = 100

print('✓ Import hoàn tất')
print(f'  BASE_DIR  : {BASE_DIR.resolve()}')
print(f'  VIHSD_DIR : {VIHSD_DIR.resolve()}')
print(f'  DATA_DIR  : {DATA_DIR.resolve()}')

✓ Import hoàn tất
  BASE_DIR  : E:\PROJECT PYTHON\New folder\toxic_analysis
  VIHSD_DIR : E:\PROJECT PYTHON\New folder\toxic_analysis\vihsd
  DATA_DIR  : E:\PROJECT PYTHON\New folder\toxic_analysis\data\collected


---
## BƯỚC 1: Thu thập & Tổng hợp dữ liệu

Dữ liệu được thu thập từ 4 nguồn:

| Nguồn | Phương pháp thu thập | File |
|-------|---------------------|------|
| **ViHSD** | Dataset có sẵn (train/dev/test) | `vihsd/train.csv`, `dev.csv`, `test.csv` |
| **Reddit** | PRAW API (r/Vietnam, r/learnvietnamese) | `reddit_comments.csv` |
| **YouTube** | YouTube Data API v3 | `youtube_comments.csv` |
| **Facebook** | Apify scraper | `facebook_comments.csv` |

In [3]:
from collect.vihsd_loader import tai_vihsd

print('▶ Tải dataset ViHSD...')
df_vihsd = tai_vihsd(vihsd_dir=str(VIHSD_DIR))

print(f'\n✓ ViHSD: {len(df_vihsd):,} mẫu')
print(f'  Cột: {df_vihsd.columns.tolist()}')
df_vihsd.head(3)

▶ Tải dataset ViHSD...


FileNotFoundError: [WinError 3] The system cannot find the path specified: 'vihsd'

In [ ]:
# ── Đọc dữ liệu thu thập từ mạng xã hội ─────────────────────────────────────
NGUON_FILES = {
    'reddit':   'reddit_comments.csv',
    'youtube':  'youtube_comments.csv',
    'facebook': 'facebook_comments.csv',
}

frames = []
for source, filename in NGUON_FILES.items():
    path = DATA_DIR / filename
    if path.exists():
        try:
            df_src = pd.read_csv(path, encoding='utf-8-sig')
            if not df_src.empty and 'text' in df_src.columns:
                df_src['source'] = source
                if 'label' not in df_src.columns:
                    df_src['label'] = None
                frames.append(df_src)
                print(f'  ✓ {source}: {len(df_src):,} bình luận')
            else:
                print(f'  ⚠ {source}: file rỗng hoặc thiếu cột text')
        except Exception as e:
            print(f'  ✗ {source}: {e}')
    else:
        print(f'  ℹ {source}: chưa có file')

df_social = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
print(f'\n✓ Tổng dữ liệu mạng xã hội: {len(df_social):,} bình luận')

In [ ]:
# ── Tổng hợp tất cả nguồn ───────────────────────────────────────────────────
df_all = pd.concat([df_vihsd, df_social], ignore_index=True)
df_all['text'] = df_all['text'].astype(str).str.strip()

print('=' * 55)
print(f'TỔNG HỢP DỮ LIỆU: {len(df_all):,} mẫu')
print('=' * 55)
print(df_all['source'].value_counts().to_string())
print()
df_all[['text', 'label', 'source']].sample(5, random_state=42)

---
## BƯỚC 2: Làm sạch dữ liệu

Quy trình làm sạch thực hiện các bước sau:

1. **Mở rộng viết tắt toxic** — `dcm` → `địt con mẹ`, `dm` → `địt mẹ`, v.v.
2. **Xóa URL, email, số điện thoại**
3. **Xóa emoji** (tùy chọn giữ lại cho phân tích cảm xúc)
4. **Rút gọn ký tự lặp** — `haaaaa` → `haa`
5. **Xóa ký tự đặc biệt**
6. **Chuẩn hóa NLP tiếng Việt** (underthesea)
7. **Xóa trùng lặp** theo (text, source)
8. **Xóa trùng gần giống** (RapidFuzz, ngưỡng 95%)

In [ ]:
from process.clean_data import lam_sach_van_ban, lam_sach_dataframe

# Demo làm sạch văn bản đơn lẻ
vi_du = [
    'dcm mày nói gì vậy??? https://example.com',
    'vl thằng kia ngu vl quá điiiiiiiii',
    'Video hay quá bạn ơi 😍😍😍!!!',
    'liên hệ: 0912345678 hoặc email@gmail.com',
]

print('Demo làm sạch văn bản:')
print('-' * 60)
for text in vi_du:
    cleaned = lam_sach_van_ban(text)
    print(f'  Gốc   : {text}')
    print(f'  Sạch  : {cleaned}')
    print()

In [ ]:
print(f'Trước làm sạch: {len(df_all):,} mẫu')

df_clean = lam_sach_dataframe(
    df_all,
    text_col='text',
    giu_emoji=False,
    xoa_trung_lap=True,
    do_dai_toi_thieu=5,
    duplicate_subset=['text', 'source'],
    similarity_threshold=95,
    similarity_group_cols=['source'],
)

print(f'Sau làm sạch : {len(df_clean):,} mẫu')
print(f'Đã loại bỏ  : {len(df_all) - len(df_clean):,} dòng')

---
## BƯỚC 3: Gán nhãn tự động bằng Gemini AI

Dữ liệu ViHSD đã có nhãn sẵn (CLEAN/OFFENSIVE/HATE). Dữ liệu từ mạng xã hội (YouTube, Reddit, Facebook) được gán nhãn tự động bằng **Gemini 2.5 Flash**.

**Quy trình incremental labeling:**
- Gửi từng batch 25 bình luận cho Gemini
- Bình luận phân loại thành công → lưu vào `labeled_collected.csv`
- Bình luận lỗi → giữ trong CSV gốc để retry
- Tự động xử lý quota limit (retry với exponential backoff)

In [ ]:
from process.label_data import phan_loai_va_di_chuyen

# Tách phần đã có nhãn (ViHSD)
df_co_nhan = df_clean[
    df_clean['label'].isin(['CLEAN', 'OFFENSIVE', 'HATE'])
].copy()

print(f'Dữ liệu đã có nhãn (ViHSD): {len(df_co_nhan):,} mẫu')

# Tải dữ liệu mạng xã hội đã gán nhãn (từ cache)
labeled_cache = DATA_DIR / 'labeled_collected.csv'
if labeled_cache.exists():
    df_labeled_social = pd.read_csv(labeled_cache, encoding='utf-8-sig')
    df_labeled_social = df_labeled_social[
        df_labeled_social['label'].isin(['CLEAN', 'OFFENSIVE', 'HATE'])
    ]
    print(f'Dữ liệu mạng xã hội đã gán nhãn: {len(df_labeled_social):,} mẫu')
    df_labeled_social.sample(min(3, len(df_labeled_social)))
else:
    df_labeled_social = pd.DataFrame()
    print('Chưa có cache gán nhãn')

In [ ]:
# ── Gộp dữ liệu đã có nhãn ──────────────────────────────────────────────────
df_labeled = pd.concat([df_co_nhan, df_labeled_social], ignore_index=True)
df_labeled = df_labeled.drop_duplicates(subset=['text', 'source'], keep='last')
df_labeled = df_labeled[df_labeled['label'].isin(['CLEAN', 'OFFENSIVE', 'HATE'])]

print('=' * 50)
print(f'TỔNG DỮ LIỆU ĐÃ GÁN NHÃN: {len(df_labeled):,} mẫu')
print('=' * 50)
print('\nPhân phối nhãn:')
label_counts = df_labeled['label'].value_counts()
for label, count in label_counts.items():
    pct = count / len(df_labeled) * 100
    print(f'  {label:12s}: {count:6,d} ({pct:.1f}%)')

print('\nPhân phối theo nguồn:')
print(df_labeled['source'].value_counts().to_string())

In [ ]:
# Xem mẫu bình luận đã gán nhãn từ mạng xã hội
print('Mẫu bình luận gán nhãn bằng Gemini AI:')
df_social_labeled = df_labeled[df_labeled['source'] != 'vihsd']
if not df_social_labeled.empty:
    display(df_social_labeled[['text', 'label', 'source']].sample(
        min(6, len(df_social_labeled)), random_state=42
    ))
else:
    print('  (Chưa có dữ liệu mạng xã hội được gán nhãn)')

---
## BƯỚC 4: Phân tích thống kê

Phân tích thống kê được thực hiện trên toàn bộ dữ liệu đã gán nhãn, gồm:

- **Phân phối nhãn** — số lượng và tỷ lệ CLEAN/OFFENSIVE/HATE
- **So sánh theo nguồn** — tỷ lệ toxic khác nhau giữa các nền tảng
- **Top từ toxic** — 20 từ xuất hiện nhiều nhất trong bình luận độc hại
- **Phân tích thời gian** — tỷ lệ toxic theo giờ trong ngày
- **Thống kê độ dài** — độ dài văn bản theo từng nhãn
- **N-gram phân tích** — bigram, trigram phổ biến trong toxic

In [ ]:
from analyze.statistics import (
    thong_ke_phan_phoi_nhan,
    so_sanh_sources,
    top_tu_toxic,
    thong_ke_do_dai,
)

# 4a. Phân phối nhãn
print('\n--- Phân phối nhãn ---')
stats_nhan = thong_ke_phan_phoi_nhan(df_labeled)

# Hiển thị bảng
df_nhan = pd.DataFrame(stats_nhan).T
df_nhan['count'] = df_nhan['count'].astype(int)
display(df_nhan)

In [ ]:
# 4b. So sánh tỷ lệ toxic theo nguồn
print('--- Tỷ lệ nhãn theo nguồn (%) ---')
ct = so_sanh_sources(df_labeled)
if not ct.empty:
    display(ct)

In [ ]:
# 4c. Top 20 từ toxic phổ biến nhất
print('--- Top 20 từ toxic ---')
df_top_words = top_tu_toxic(df_labeled, top_n=20)
display(df_top_words.head(10))

In [ ]:
# 4d. Thống kê độ dài văn bản
print('--- Thống kê độ dài văn bản theo nhãn ---')
df_dodo = thong_ke_do_dai(df_labeled)
display(df_dodo)

In [ ]:
from analyze.advanced_stats import chay_phan_tich_nang_cao

print('--- Phân tích nâng cao (N-gram, Toxic Intensity, Correlation, Topic) ---')
advanced_stats = chay_phan_tich_nang_cao(df_labeled)

if advanced_stats and 'bigrams' in advanced_stats:
    print('\nTop 10 Bigram Toxic:')
    df_bi = pd.DataFrame(advanced_stats['bigrams'][:10], columns=['bigram', 'count'])
    display(df_bi)

---
## BƯỚC 5: Trực quan hóa kết quả

Tất cả biểu đồ được lưu vào `output/charts/`. Pipeline tạo 13 biểu đồ:

| # | Tên biểu đồ | Mô tả |
|---|-------------|-------|
| 01 | Phân phối nhãn | Biểu đồ cột CLEAN/OFFENSIVE/HATE |
| 02 | Tỷ lệ toxic | Biểu đồ tròn toxic vs clean |
| 03 | Top từ toxic | Top 20 từ độc hại phổ biến nhất |
| 04 | WordCloud | WordCloud riêng cho HATE/OFFENSIVE/CLEAN |
| 05 | Toxic theo giờ | Tỷ lệ toxic theo giờ trong ngày |
| 06 | So sánh nguồn | Toxic theo nền tảng |
| 07 | Bigram | Cụm 2 từ phổ biến trong toxic |
| 08 | Trigram | Cụm 3 từ phổ biến trong toxic |
| 09 | Mức độ toxic | Phân phối mức độ độc hại |
| 10 | Tương quan | Độ dài văn bản vs mức độ toxic |
| 11 | Chủ đề | Phân cụm chủ đề theo nhãn |

In [ ]:
from analyze.statistics import chay_tat_ca_thong_ke
from analyze.visualize import ve_tat_ca_bieu_do
from analyze.advanced_visualize import ve_tat_ca_bieu_do_nang_cao

# Chạy thống kê
stats = chay_tat_ca_thong_ke(df_labeled)

# Tạo biểu đồ
CHARTS_DIR.mkdir(parents=True, exist_ok=True)
saved_charts = ve_tat_ca_bieu_do(df_labeled, stats, output_dir=str(CHARTS_DIR))
if advanced_stats:
    saved_adv = ve_tat_ca_bieu_do_nang_cao(advanced_stats, output_dir=str(CHARTS_DIR))
    saved_charts.extend(saved_adv)

print(f'✓ Đã tạo {len(saved_charts)} biểu đồ → {CHARTS_DIR}')

In [ ]:
# ── Hiển thị biểu đồ 01: Phân phối nhãn ─────────────────────────────────────
chart_01 = CHARTS_DIR / '01_phan_phoi_nhan.png'
if chart_01.exists():
    display(Image(filename=str(chart_01)))

In [ ]:
# ── Hiển thị biểu đồ 02: Tỷ lệ toxic ───────────────────────────────────────
chart_02 = CHARTS_DIR / '02_ty_le_toxic.png'
if chart_02.exists():
    display(Image(filename=str(chart_02)))

In [ ]:
# ── Hiển thị biểu đồ 03: Top từ toxic ──────────────────────────────────────
chart_03 = CHARTS_DIR / '03_top_tu_toxic.png'
if chart_03.exists():
    display(Image(filename=str(chart_03)))

In [ ]:
# ── Hiển thị biểu đồ 04: WordCloud ──────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
wc_files = {
    'HATE': '04_wordcloud_hate.png',
    'OFFENSIVE': '04_wordcloud_offensive.png',
    'CLEAN': '04_wordcloud_clean.png',
}
for ax, (label, fname) in zip(axes, wc_files.items()):
    path = CHARTS_DIR / fname
    if path.exists():
        img = mpimg.imread(str(path))
        ax.imshow(img)
        ax.axis('off')
        ax.set_title(f'WordCloud — {label}', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Hiển thị biểu đồ 05 & 06 ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for ax, fname in zip(axes, ['05_toxic_theo_gio.png', '06_so_sanh_nguon.png']):
    path = CHARTS_DIR / fname
    if path.exists():
        img = mpimg.imread(str(path))
        ax.imshow(img)
        ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# ── Hiển thị biểu đồ nâng cao 07–11 ────────────────────────────────────────
adv_charts = [
    '07_bigram_toxic.png', '08_trigram_toxic.png',
    '09_muc_do_toxic.png', '10_tuong_quan_dodai_toxic.png',
    '11_chu_de_theo_nhan.png'
]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes_flat = axes.flatten()

for i, fname in enumerate(adv_charts):
    path = CHARTS_DIR / fname
    if path.exists():
        img = mpimg.imread(str(path))
        axes_flat[i].imshow(img)
        axes_flat[i].axis('off')
        axes_flat[i].set_title(fname.replace('.png', '').replace('_', ' ').title(), fontsize=10)

# Ẩn ô trống
for j in range(len(adv_charts), len(axes_flat)):
    axes_flat[j].set_visible(False)

plt.tight_layout()
plt.show()

---
## Tổng kết kết quả

### Tóm tắt số liệu

In [ ]:
# ── Tải báo cáo tổng hợp ────────────────────────────────────────────────────
summary_path = OUTPUT_DIR / 'summary.json'
if summary_path.exists():
    with open(summary_path, encoding='utf-8') as f:
        summary = json.load(f)

    print('=' * 60)
    print('BÁO CÁO TỔNG HỢP KẾT QUẢ PHÂN TÍCH')
    print('=' * 60)
    print(f'\nTổng mẫu phân tích  : {summary["total_rows"]:,}')

    print('\nPhân phối nhãn:')
    for label, info in summary['label_distribution'].items():
        bar = '█' * int(info['pct'] / 2)
        print(f'  {label:12s}: {info["count"]:6,d} ({info["pct"]:5.1f}%) {bar}')

    print('\nPhân phối theo nguồn:')
    for source, count in summary['source_distribution'].items():
        print(f'  {source:12s}: {count:6,d}')

    print('\nTỷ lệ toxic theo nền tảng:')
    for row in summary['toxic_by_source']:
        print(f'  {row["source"]:12s}: {row["toxic_rate_pct"]:5.1f}% '
              f'({row["toxic"]}/{row["total"]} bình luận)')

    print(f'\nSố biểu đồ đã tạo   : {len(summary["charts"])}')
    print(f'Kết quả CSV          : output/results.csv')
    print(f'Tóm tắt JSON         : output/summary.json')

In [ ]:
# ── Lưu kết quả cuối cùng ───────────────────────────────────────────────────
results_path = OUTPUT_DIR / 'results.csv'
cols_out = [c for c in ['text', 'label', 'source'] if c in df_labeled.columns]
df_labeled[cols_out].to_csv(results_path, index=False, encoding='utf-8-sig')
print(f'✓ Đã lưu {len(df_labeled):,} mẫu → {results_path}')

# Xem 5 dòng đầu
df_labeled[cols_out].head()

---
## Kết luận

### Phát hiện chính

1. **Tỷ lệ toxic tổng thể** — Khoảng **18.66%** bình luận có nội dung độc hại (OFFENSIVE + HATE), trong đó HATE (11.43%) nhiều hơn OFFENSIVE (7.23%).

2. **Chênh lệch giữa các nền tảng** — Reddit có tỷ lệ toxic cao nhất (**51.66%**), tiếp theo là ViHSD (**18.2%**). Facebook và YouTube trong tập dữ liệu thu thập có tỷ lệ thấp hơn do lượng mẫu còn hạn chế.

3. **Từ khóa toxic phổ biến** — Các từ viết tắt và biến thể chữ viết của ngôn ngữ thô tục tiếng Việt xuất hiện với tần suất cao trong nhóm OFFENSIVE/HATE.

4. **Độ dài văn bản** — Bình luận HATE có xu hướng dài hơn OFFENSIVE và CLEAN, cho thấy nội dung thù ghét thường được diễn đạt chi tiết hơn.

### Hạn chế

- Dữ liệu Facebook và YouTube thu thập được còn ít (74 và 14 mẫu), chưa đủ đại diện.
- Việc gán nhãn tự động bằng Gemini AI có thể có sai sót trong các trường hợp ngữ cảnh phức tạp.
- Dataset ViHSD chiếm tỷ lệ lớn (98.2%), cần bổ sung thêm dữ liệu thực tế từ mạng xã hội.

### Hướng phát triển

- Thu thập thêm dữ liệu từ Facebook, YouTube, TikTok
- Huấn luyện mô hình phân loại riêng (PhoBERT, ViSoBERT)
- Phân tích ngữ cảnh và chuỗi bình luận (thread analysis)
- Xây dựng công cụ lọc bình luận toxic real-time